# fake_pico.py
**網路、多人遊戲與 Web 全端 › 🚀 延伸:MQTT 捷徑路線(聊天室、多人遊戲、Pico 即時儀表板)**

沒有 Pico 也能玩:這支 Python 假裝成一片 Pico,發布模擬的感測器數據。資料格式和 Pico 一模一樣,所以儀表板分不出真假。這正好示範發布訂閱的好處:訂閱的一方不知道、也不在乎是誰發的,只要主題和格式對就好。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/5)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!pip -q install paho-mqtt
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), 'fake_pico.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 說明與匯入
開頭說明它的角色:假裝成一片 Pico。匯入的 math 用來做出擺動的波形,random 加雜訊和產生裝置名稱,paho-mqtt 則和聊天室、遊戲伺服器用的是同一個套件。

In [ ]:
# 沒有 Pico 也能玩:用 Python 假裝成一片 Pico,發布「模擬的」MPU6050 數據
# 格式跟 pico2w_mqtt_mpu6050.ino 一模一樣 → dashboard.html 分不出真假
# 安裝:pip install paho-mqtt
# 執行:python fake_pico.py [房間名稱]

import json
import math
import random
import sys
import time

import paho.mqtt.client as mqtt

### 設定與連線
裝置名稱是 fake 加四個隨機十六進位字,主題格式和 Pico 相同:房間、pico、裝置名稱。先設遺囑,關掉程式網頁就顯示離線;連上後發布保留的 online。這幾行你在聊天室和遊戲伺服器都看過,MQTT 程式的開頭幾乎都長這樣。

In [ ]:
ROOM   = sys.argv[1] if len(sys.argv) > 1 else "demo"
DEVICE = "fake-" + "".join(random.choice("0123456789abcdef") for _ in range(4))
TOPIC  = f"pyl-academy/v1/{ROOM}/pico/{DEVICE}/"

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=DEVICE)
client.will_set(TOPIC + "status", "offline", retain=True)       # 關掉程式 → 網頁顯示離線
client.connect("broker.emqx.io", 1883, keepalive=15)
client.loop_start()
client.publish(TOPIC + "status", "online", retain=True)
print(f"模擬裝置 {DEVICE} 開始發布到房間「{ROOM}」,打開 dashboard.html?room={ROOM} 看曲線(Ctrl+C 結束)")

### 產生模擬數據
這裡的數學只是例子,重點是格式和 Pico 一樣。用 sin 和 cos 做出來回擺動的樣子,假裝板子在桌上慢慢搖;z 軸固定約 1g,那是重力;再用 gauss 加一點隨機雜訊,看起來更像真的感測器。t 是經過的毫秒數。

In [ ]:
t0 = time.time()

### 發布並等待
把資料轉成 JSON 發布到 imu 主題,然後睡零點一秒,也就是每秒十筆,和真的 Pico 一樣。感測資料不設保留,因為一直在更新。

### 收尾
按 Ctrl 加 C 結束時不印錯誤訊息,finally 主動發布 offline,等一下讓它送出,再停止背景執行緒並斷線。這個收尾模式和前面兩支 Python 程式完全一樣。

In [ ]:
try:
    while True:
        t = time.time() - t0
        # 假裝板子在桌上慢慢搖晃:z 軸約 1g(重力),x、y 小幅擺動,再加一點雜訊
        data = {
            "ax": round(0.3 * math.sin(t * 1.5) + random.gauss(0, 0.02), 3),
            "ay": round(0.2 * math.sin(t * 0.7 + 1) + random.gauss(0, 0.02), 3),
            "az": round(1 + random.gauss(0, 0.02), 3),
            "gx": round(45 * math.cos(t * 1.5), 1),
            "gy": round(14 * math.cos(t * 0.7 + 1), 1),
            "gz": round(random.gauss(0, 1), 1),
            "t": int(t * 1000),
        }
        client.publish(TOPIC + "imu", json.dumps(data))
        time.sleep(0.1)                                         # 每秒 10 筆,跟真的 Pico 一樣
except KeyboardInterrupt:
    pass
finally:
    client.publish(TOPIC + "status", "offline", retain=True)
    time.sleep(0.3)
    client.loop_stop()
    client.disconnect()

## 小結
模擬器和真的硬體發布同樣的主題、同樣的格式,接收端就能共用。開發時先用模擬器把網頁做好,硬體到了再換上,這是很實用的工作方式。下一支 dashboard.html 把這些數據畫成即時曲線。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/5) 看逐句拆解、做練習。